In [2]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# Clone the repository to this location, or change REPO_ROOT to your checkout.
REPO_ROOT = Path("/content/drive/MyDrive/jev-like-decision-tuning")
assert (REPO_ROOT / "pyproject.toml").is_file(), "Set REPO_ROOT to the repository checkout"
%pip install -q -e "$REPO_ROOT"

from jev_decision_tuning.data import (
    load_sentiment_dataset,
    sample_examples,
    sample_balanced,
    format_training_dataset,
    tokenize_dataset,
)
from jev_decision_tuning.prompts import (
    INSTRUCTION,
    LABEL_NAMES,
    LETTERS,
    build_prompt,
    format_example,
    render_chat_prompt,
)

label_names = list(LABEL_NAMES)
letters = list(LETTERS)

ModuleNotFoundError: No module named 'google'

In [ ]:
ds = load_sentiment_dataset(
    "/content/drive/MyDrive/jev_project/data/tweet_eval_sentiment"
)

## Format a sentiment example

In [ ]:
instance = ds["train"][0]
result = format_example(
    instance,
    compact_options=True,
    include_reply_instruction=False,
)
print(result["prompt"])
print(result["target_letter"])

In [ ]:
%pip install -q -U transformers accelerate

## Load the model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto",
)

model.eval()

print("Model device:", model.device)

## Evaluate the base model on 100 validation examples

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

# 1. Fix a 100-example subset for before/after comparisons.
samples = sample_examples(ds["validation"], count=100, seed=42)

labels = list(LABEL_NAMES)

encoded = [
    tokenizer.encode(letter, add_special_tokens=False)
    for letter in letters
]
assert all(len(ids) == 1 for ids in encoded)
option_ids = [ids[0] for ids in encoded]


# 2. Score one example with the original compact option spacing.
def predict_one(example):
    prompt = build_prompt(example["text"], compact_options=True)
    text = render_chat_prompt(tokenizer, prompt)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model(**inputs)
        scores = outputs.logits[0, -1, option_ids].float()
        probs = torch.softmax(scores, dim=-1).cpu().tolist()

    return probs


# 3. Run predictions and retain the results.
model.eval()
records = []

for example in tqdm(samples):
    probs = predict_one(example)
    prediction = max(range(3), key=lambda i: probs[i])

    records.append({
        "text": example["text"],
        "true_label": example["label"],
        "pred_label": prediction,
        "p_negative": probs[0],
        "p_neutral": probs[1],
        "p_positive": probs[2],
    })

results = pd.DataFrame(records)

# 4. Compute metrics.
y_true = results["true_label"]
y_pred = results["pred_label"]

print(f"\nAccuracy:{accuracy_score(y_true, y_pred):.1%}")

print("\nPer-class performance:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    target_names=labels,
    digits=3,
    zero_division=0,
))

print("Confusion matrix (rows=true, columns=predicted):")
display(pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=[0, 1, 2]),
    index=[f"true_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels],
))

## Fine-tune the model with LoRA

In [ ]:
from collections import Counter

# Sample 3,000 examples per class: 9,000 training examples in total.
train_small = sample_balanced(ds["train"], per_class=3000, seed=42)

print("Label order:", label_names)
print("Training examples:", len(train_small))
counts = Counter(train_small["label"])
for label_id, label_name in enumerate(label_names):
    print(f"{label_name}: {counts[label_id]}")
print("First example:", train_small[0])

In [ ]:
# Option shuffling and target remapping use a fresh local RNG on each call.
train_formatted = format_training_dataset(train_small, seed=42)

for i in range(3):
    example = train_formatted[i]
    print(example["prompt"])
    print("Correct option:", example["target_letter"])
    print("Original class:", label_names[example["label"]])
    print()

In [ ]:
train_tokenized = tokenize_dataset(train_formatted, tokenizer)

lengths = [len(ids) for ids in train_tokenized["input_ids"]]
print("Examples:", len(train_tokenized))
print("Maximum token count:", max(lengths))
print("Mean token count:", round(sum(lengths) / len(lengths), 1))
print("First decoded input:")
print(tokenizer.decode(train_tokenized[0]["input_ids"]))
print("Correct option:", letters[train_tokenized[0]["labels"]])

In [ ]:
%pip install -q -U peft

In [ ]:
%pip uninstall -y torchao

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import set_seed

set_seed(42)

target_modules = [
    # Attention modules
    "q_proj", "k_proj", "v_proj", "o_proj",

    # MLP modules
    "gate_proj", "up_proj", "down_proj",
]

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    target_modules=target_modules,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False

model.print_trainable_parameters()

In [ ]:
import torch
from torch.utils.data import DataLoader

# Right-pad inputs of different lengths.
tokenizer.padding_side = "right"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def collate_batch(examples):
    # Keep the target answer positions separate.
    labels = torch.tensor(
        [example["labels"] for example in examples],
        dtype=torch.long,
    )

    # Collect model inputs.
    features = [
        {
            "input_ids": example["input_ids"],
            "attention_mask": example["attention_mask"],
        }
        for example in examples
    ]

    # Pad inputs to the same length within this batch.
    batch = tokenizer.pad(
        features,
        padding=True,
        return_tensors="pt",
    )

    batch["labels"] = labels
    return batch


train_loader = DataLoader(
    train_tokenized,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_batch,
    generator=torch.Generator().manual_seed(42),
)

batch = next(iter(train_loader))

print("Input shape:", batch["input_ids"].shape)
print("Target option indices:", batch["labels"])
print("Actual input lengths:", batch["attention_mask"].sum(dim=1))

In [ ]:
print("Training examples:", len(train_tokenized))
print("Batch size:", train_loader.batch_size)
print("Total batches:", len(train_loader))
model.print_trainable_parameters()

In [ ]:
import json
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import set_seed

set_seed(42)
device = model.device

# Use a separate output directory for each run.
save_dir = (
    Path("/content/drive/MyDrive/jev_project")
    / f"lora_attention_mlp_{time.strftime('%Y%m%d_%H%M%S')}"
)

assert Path("/content/drive/MyDrive").is_dir(), "Mount Google Drive first"
save_dir.mkdir(parents=True, exist_ok=False)

# Train only LoRA parameters and keep trainable parameters in float32.
trainable_params = []

for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        assert "lora_" in name, f"Non-LoRA parameter is trainable:{name}"
        parameter.data = parameter.data.float()
        trainable_params.append(parameter)

# Token IDs for A, B, and C
encoded_options = [
    tokenizer.encode(letter, add_special_tokens=False)
    for letter in ["A", "B", "C"]
]
assert all(len(ids) == 1 for ids in encoded_options)

option_ids = torch.tensor(
    [ids[0] for ids in encoded_options],
    device=device,
)

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=5e-5,
    weight_decay=0.01,
)
scaler = torch.amp.GradScaler("cuda", init_scale=1024)

# Reset shuffling so inspecting a batch does not change training order.
train_loader.generator.manual_seed(42)
assert tokenizer.padding_side == "right"

model.config.use_cache = False
model.train()

history = []
window_losses = []
skipped_updates = 0

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
start = time.perf_counter()

progress = tqdm(train_loader, desc="Training epoch 1/1")

for step, batch in enumerate(progress, start=1):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    targets = batch["labels"].to(device)

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast("cuda", dtype=torch.float16):
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )

        # Select the last non-padding token for each input.
        last_positions = attention_mask.sum(dim=1) - 1
        row_ids = torch.arange(input_ids.shape[0], device=device)

        scores = outputs.logits[row_ids, last_positions]
        option_scores = scores[:, option_ids].float()

        # Targets are A/B/C positions; compute cross-entropy over the three options.
        loss = F.cross_entropy(option_scores, targets)

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at batch {step}; stopping training")

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)

    old_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()

    skipped = scaler.get_scale() < old_scale
    skipped_updates += int(skipped)

    loss_value = loss.item()
    history.append({
        "step": step,
        "loss": loss_value,
        "update_skipped": skipped,
    })
    window_losses.append(loss_value)

    # Report the mean loss over each 50-batch window.
    if step % 50 == 0:
        mean_loss = sum(window_losses) / len(window_losses)
        tqdm.write(f"Batch {step}/{len(train_loader)} | Mean loss over last 50 batches:{mean_loss:.4f}")
        window_losses.clear()

    del outputs, scores, option_scores, loss

optimizer.zero_grad(set_to_none=True)
model.eval()

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

# Save the trained LoRA adapter and tokenizer.
model.save_pretrained(str(save_dir))
tokenizer.save_pretrained(str(save_dir))

# Save the experiment configuration and loss history.
record = {
    "base_model": "Qwen/Qwen3-1.7B",
    "seed": 42,
    "samples": len(train_tokenized),
    "batch_size": train_loader.batch_size,
    "epochs": 1,
    "learning_rate": 5e-5,
    "weight_decay": 0.01,
    "loss": "cross_entropy_over_A_B_C",
    "skipped_updates": skipped_updates,
    "elapsed_seconds": elapsed,
    "history": history,
}

with open(save_dir / "training_log.json", "w", encoding="utf-8") as f:
    json.dump(record, f, ensure_ascii=False, indent=2)

print(f"\nTraining completed in {elapsed / 60:.1f} minutes")
print(f"Skipped updates:{skipped_updates}")
print(f"Peak GPU memory:{torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")
print("Saved to:", save_dir)

In [ ]:
%pip install -q typesafe-sdk

In [ ]:
import json
from pathlib import Path

import torch
import pandas as pd
from tqdm.auto import tqdm
from google.colab import userdata
from typesafe_sdk import Choice, TypeSafeClient
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    confusion_matrix,
)

instruction = INSTRUCTION

save_dir = Path("/content/drive/MyDrive/jev_project")/ f"lora_attention_mlp_{time.strftime('%Y%m%d_%H%M%S')}"
jev_dir = Path("/content/drive/MyDrive/jev_project/jev_1.13.0_validation")
assert Path("/content/drive/MyDrive").is_dir(), "Mount Google Drive first"
save_dir.mkdir(parents=True, exist_ok=True)
jev_dir.mkdir(parents=True, exist_ok=True)

encoded = [
    tokenizer.encode(letter, add_special_tokens=False)
    for letter in letters
]
assert all(len(ids) == 1 for ids in encoded)
option_ids = [ids[0] for ids in encoded]

In [ ]:
def evaluate_validation(
    description,
    backend="qwen",
    dataset=None,
    checkpoint_name="predictions",
):
    """
    backend="qwen": use the current local model.
    backend="jev": call the official Jev API.
    """
    assert backend in ["qwen", "jev"]

    validation = ds["validation"] if dataset is None else dataset
    assert validation.features["label"].names == label_names
    records = {}

    checkpoint = jev_dir / f"{checkpoint_name}.jsonl"

    # Resume Jev evaluation from saved predictions.
    if backend == "jev" and checkpoint.exists():
        with checkpoint.open(encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue

                record = json.loads(line)
                row_id = record["row_id"]

                assert 0 <= row_id < len(validation)
                assert record["text"] == validation[row_id]["text"]
                assert record["true_label"] == validation[row_id]["label"]
                assert record["model"] == "jev-1.13.0"

                records[row_id] = record

    def save_record(row_id, example, probs, **extra):
        record = {
            "row_id": row_id,
            "text": example["text"],
            "true_label": example["label"],
            "pred_label": max(range(3), key=lambda i: probs[i]),
            "p_negative": probs[0],
            "p_neutral": probs[1],
            "p_positive": probs[2],
            **extra,
        }
        records[row_id] = record
        return record

    if backend == "qwen":
        model.eval()

        for row_id, example in enumerate(
            tqdm(validation, desc=description)
        ):
            prompt = build_prompt(example["text"])
            text = render_chat_prompt(tokenizer, prompt)

            inputs = tokenizer(
                text, return_tensors="pt"
            ).to(model.device)

            with torch.inference_mode():
                outputs = model(**inputs, use_cache=False)
                scores = outputs.logits[0, -1, option_ids].float()
                probs = torch.softmax(scores, dim=-1).cpu().tolist()

            save_record(row_id, example, probs)
            del outputs

    else:
        with TypeSafeClient(api_key=userdata.get("jev")) as client:
            with checkpoint.open("a", encoding="utf-8") as f:
                with tqdm(
                    total=len(validation),
                    initial=len(records),
                    desc=description,
                ) as progress:

                    for row_id, example in enumerate(validation):
                        if row_id in records:
                            continue

                        response = client.system_one(
                            model="jev-1.13.0",
                            state=example["text"],
                            questions={
                                "sentiment": Choice(
                                    instructions=instruction,
                                    criteria=dict(zip(LETTERS, LABEL_NAMES)),
                                ),
                            },
                        )

                        assert response.model == "jev-1.13.0"
                        answer = response.choices["sentiment"]

                        probs = [
                            float(answer.probabilities[letter])
                            for letter in letters
                        ]

                        record = save_record(
                            row_id,
                            example,
                            probs,
                            api_choice=answer.choice,
                            model=response.model,
                            input_tokens=response.usage.input_tokens,
                            output_tokens=response.usage.output_tokens,
                        )

                        f.write(
                            json.dumps(record, ensure_ascii=False) + "\n"
                        )
                        f.flush()
                        progress.update(1)

    assert set(records) == set(range(len(validation)))

    return pd.DataFrame(
        [records[i] for i in range(len(validation))]
    )

In [ ]:
# First, disable LoRA and evaluate the base model.
with model.disable_adapter():
    results_base = evaluate_validation("Base model", backend="qwen")

results_base.to_csv(
    save_dir / "validation_full_base.csv", index=False
)

# Second, restore LoRA and evaluate the fine-tuned model.
results_lora = evaluate_validation("LoRA model", backend="qwen")

results_lora.to_csv(
    save_dir / "validation_full_lora.csv", index=False
)

# Third, evaluate the official Jev model.
results_jev = evaluate_validation("Jev model", backend="jev")

results_jev.to_csv(
    jev_dir / "validation_full_jev.csv", index=False
)

In [ ]:
summary = []

for name, frame in [
    ("Base model", results_base),
    ("LoRA model", results_lora),
    ("Jev 1.13.0", results_jev),
]:
    assert frame["row_id"].tolist() == results_base["row_id"].tolist()
    assert frame["true_label"].tolist() == results_base["true_label"].tolist()

    y_true = frame["true_label"]
    y_pred = frame["pred_label"]

    recalls = recall_score(
        y_true, y_pred,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )

    summary.append({
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro-F1": f1_score(
            y_true, y_pred,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        ),
        "negative_recall": recalls[0],
        "neutral_recall": recalls[1],
        "positive_recall": recalls[2],
    })

    print(f"\n{name}: rows=true, columns=predicted")
    display(pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=[0, 1, 2]),
        index=label_names,
        columns=label_names,
    ))

summary_df = pd.DataFrame(summary).set_index("model")

print("\nFull validation comparison across three models:")
display(summary_df.round(4))

summary_df.to_csv(
    save_dir / "validation_comparison_with_jev.csv"
)

In [ ]:
# Keep LoRA enabled for training and test split evaluation.
assert len(train_small) == 9000

save_dir.mkdir(parents=True, exist_ok=True)

results_train_lora = evaluate_validation(
    "LoRA / train",
    backend="qwen",
    dataset=train_small,
)
results_train_lora.to_csv(
    save_dir / "train_lora.csv", index=False
)

results_test_lora = evaluate_validation(
    "LoRA / test",
    backend="qwen",
    dataset=ds["test"],
)
results_test_lora.to_csv(
    save_dir / "test_lora.csv", index=False
)

In [ ]:
summary = []

for split, frame in [
    ("train", results_train_lora),
    ("validation", results_lora),
    ("test", results_test_lora),
]:
    y_true = frame["true_label"]
    y_pred = frame["pred_label"]

    recalls = recall_score(
        y_true, y_pred,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )

    summary.append({
        "split": split,
        "examples": len(frame),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro-F1": f1_score(
            y_true, y_pred,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        ),
        "negative_recall": recalls[0],
        "neutral_recall": recalls[1],
        "positive_recall": recalls[2],
    })

split_summary = pd.DataFrame(summary).set_index("split")
display(split_summary.round(4))

split_summary.to_csv(save_dir / "lora_split_comparison.csv")